In [3]:
!pip uninstall -y torchvision torchaudio
!pip install -q torch~=2.3.0 transformers datasets accelerate peft evaluate rouge_score pandas


Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126


In [6]:
!pip install opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: f223644saifullah
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail


100%|██████████| 503M/503M [00:05<00:00, 104MB/s]


In [7]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Define sample sizes
n_train = 50000
n_validation = 5000
n_test = 5000

try:
    # Load the full datasets and then take a random sample from each.
    # Using random_state to ensure reproducibility.
    train_df = pd.read_csv('/content/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/train.csv').sample(n=n_train, random_state=42)
    validation_df = pd.read_csv('/content/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/validation.csv').sample(n=n_validation, random_state=42)
    test_df = pd.read_csv('/content/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/test.csv').sample(n=n_test, random_state=42)

    # Create Hugging Face Datasets
    train_dataset = Dataset.from_pandas(train_df)
    validation_dataset = Dataset.from_pandas(validation_df)
    test_dataset = Dataset.from_pandas(test_df)

    # Combine them into a DatasetDict
    raw_datasets = DatasetDict({
        'train': train_dataset,
        'validation': validation_dataset,
        'test': test_dataset
    })

    print("Datasets loaded successfully with random sampling:")
    print(raw_datasets)
    print("\nExample article:", raw_datasets['train'][0]['article'])
    print("\nExample summary:", raw_datasets['train'][0]['highlights'])

except FileNotFoundError:
    print("Make sure 'train.csv', 'validation.csv', and 'test.csv' are in the same directory.")
except ValueError:
    print(f"One of the files is smaller than the requested sample size. Please check the number of rows in your CSV files.")

Datasets loaded successfully with random sampling:
DatasetDict({
    train: Dataset({
        features: ['id', 'article', 'highlights', '__index_level_0__'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['id', 'article', 'highlights', '__index_level_0__'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'article', 'highlights', '__index_level_0__'],
        num_rows: 5000
    })
})

Example article: By . Mia De Graaf . Britons flocked to beaches across the southern coast yesterday as millions look set to bask in glorious sunshine today. Temperatures soared to 17C in Brighton and Dorset, with people starting their long weekend in deck chairs by the sea. Figures from Asda suggest the unexpected sunshine has also inspired a wave of impromptu barbecues, with sales of sausages and equipment expected to triple those in April. Sun's out: Brighton beach was packed with Britons enjoying the unexpected sunshine to start the long weekend as

In [8]:
from transformers import AutoTokenizer

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# T5 requires a prefix for summarization tasks
prefix = "summarize: "

def preprocess_function(examples):
    # Handle potential None values
    inputs = [prefix + doc for doc in examples["article"] if doc is not None]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")

    # Setup the tokenizer for targets
    labels = [summ for summ in examples["highlights"] if summ is not None]
    labels = tokenizer(text_target=labels, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing to all datasets
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, remove_columns=['id', 'article', 'highlights'])
print(tokenized_datasets)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
})


In [9]:
import torch
import os
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

# --- GPU/CPU Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# Load the base T5 model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

# Define LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"], # Target attention modules for T5
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM # This is a sequence-to-sequence model
)

# Wrap the model with PEFT
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

Running on: cuda


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 589,824 || all params: 61,096,448 || trainable%: 0.9654


In [ ]:
import transformers
import numpy as np
import evaluate
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from pathlib import Path

# Load ROUGE metric
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode generated summaries and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Add mean generated length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# Check if model already exists
model_save_path = "t5-summarizer-final"
if Path(model_save_path).exists():
    print(f"✅ Model already exists at '{model_save_path}'")
    print("Loading the existing model instead of training...")
    
    from peft import PeftModel
    
    # Load the base model
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    # Load the PEFT model with saved adapter
    peft_model = PeftModel.from_pretrained(base_model, model_save_path)
    peft_model.to(device)
    print("✅ Model loaded successfully!")
    
else:
    print(f"⚙️ Model not found. Starting training...")
    
    # --- GPU-Specific Training Arguments ---
    training_args = Seq2SeqTrainingArguments(
        output_dir="t5-summarizer",
        num_train_epochs=3,
        per_device_train_batch_size=4, # Adjust as needed
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=3e-4,
        fp16=True, # Use fp16 for mixed-precision training on GPU
        predict_with_generate=True, # Enable summary generation during evaluation
        logging_dir='./logs',
        logging_steps=50,
        save_strategy="epoch",      # Corrected: was save_strategy
        eval_strategy="epoch",      # Corrected: was evaluation_strategy
    )

    # Data collator for sequence-to-sequence tasks
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    # Create the Trainer
    trainer = Seq2SeqTrainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # Start training
    trainer.train()
    
    # Save the final model to the output directory
    trainer.save_model(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    print(f"✅ Model trained and saved to '{model_save_path}'")


/tmp/ipython-input-3879057893.py:46: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: su3561116 (su3561116-wqxr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Save the final model to the output directory
trainer.save_model("t5-summarizer-final")

# You can also save the tokenizer
tokenizer.save_pretrained("t5-summarizer-final")

In [ ]:
# Take a sample from the test set
sample = raw_datasets['test'][0]
input_text = prefix + sample['article']

# Set device for inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Tokenize the input
inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to(device)

# Generate the summary
peft_model.eval()
with torch.no_grad():
    outputs = peft_model.generate(
        input_ids=inputs["input_ids"],
        max_new_tokens=128,
        do_sample=True,
        top_p=0.9,
        temperature=0.7
    )

generated_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Original Article ---")
print(sample['article'])
print("\n--- Reference Summary ---")
print(sample['highlights'])
print("\n--- Generated Summary (Fine-tuned T5) ---")
print(generated_summary)